In [ ]:
import torch
from torch.utils import data
import os
import json
from datetime import datetime

from datasets import ImageNet, Places365, ArtPlaces, iNaturalist

from evaluation_utils import create_model, extract_and_evaluate_features

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print(torch.cuda.get_device_name())

### Konstanten

Importieren der nötigen Konstanten aus `evaluation_constants.py`

- `DEST`: Speicherort der Ergebnisse
- `DISTANCE`: Definition der Distanzmetrik für den faiss Index (`l2` oder `cosine`)
- `K`: Definition der Ks (Liste von Zahlen)
- `DATASETS`: Definition der Datensätze (Liste von Strings, `ImageNet`, `Places365` oder `ArtPlaces`)

In [ ]:
from evaluation_constants import DEST, DISTANCE, K, MODELS, DATASETS

### Datensätze

Laden der Datensätze und erstellen der Dataloader

In [ ]:
BATCH_SIZE = 64
NUM_WORKERS = 8

imagenet = ImageNet(root=r"D:\Dokumente\Studium\Praktikum\ImageNet_Evaluation_Subset")
imagenet_dataloader = data.DataLoader(imagenet, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

places365 = Places365(root=r"D:\Dokumente\Studium\Praktikum\Places365_Evaluation_Subset")
places365_dataloader = data.DataLoader(places365, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

artplaces = ArtPlaces(root=r"C:\Users\mariu\Documents\Development\Datasets\ArtPlaces_13371280")
artplaces_dataloader = data.DataLoader(artplaces, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

inaturalist_family = iNaturalist(root=r"D:\Dokumente\Development\Datasets\iNaturalist", split="train_mini", target_type="family")
inaturalist_family_dataloader = data.DataLoader(inaturalist_family, batch_size=BATCH_SIZE)

datasets = {
    "ImageNet": imagenet,
    "ImageNet_Loader": imagenet_dataloader,
    "Places365": places365,
    "Places365_Loader": places365_dataloader,
    "ArtPlaces": artplaces,
    "ArtPlaces_Loader": artplaces_dataloader,
    "iNaturalist_Family": inaturalist_family,
    "iNaturalist_Family_Loader": inaturalist_family_dataloader,
}

### Durchführen der Experimente

Für jede Methode und jeden Datensatz werden die folgenden Schritte durchgeführt:
1. Extraktion der Merkmale
2. Einfügen der Features in einen Faiss-Index
3. Suchen der n nächsten Nachbarn für jedes Bild eines Datensatzes
4. Bewerten der gefunden Bilder mithilfe verschiedener Metriken
5. Einfügen der Ergebnisse in das results Dictionary

In [ ]:
# for m in MODELS:
#     model = create_model(m, device)

#     _, (b, l) = next(enumerate(imagenet_dataloader))
#     b = b.to(device)

#     print(f"{m["name"]}: {model(b).shape}")

In [ ]:
results = {}

for m in MODELS:
    results[m["name"]] = {}
    model = create_model(m)

    for d in DATASETS:
        result = extract_and_evaluate_features(model, m["name"], datasets[d + "_Loader"], d, DISTANCE, K)
        results[m["name"]][d] = result

In [ ]:
dest = os.path.join(DEST, datetime.today().strftime('%Y%m%d_%H%M%S'))

os.makedirs(dest)

with open(os.path.join(dest, "results.json"), mode="w+") as file:
    json.dump(results, file)

with open(os.path.join(dest, "DISTANCE.json"), mode="w+") as file:
    json.dump(DISTANCE, file)

with open(os.path.join(dest, "K.json"), mode="w+") as file:
    json.dump(K, file)

with open(os.path.join(dest, "DATASETS.json"), mode="w+") as file:
    json.dump(DATASETS, file)

with open(os.path.join(dest, "MODELS.json"), mode="w+") as file:
    json.dump(MODELS, file)